# Smoke test — pinned MIT BEVFusion (`da93b27`) + official fused checkpoint

Ticket 02 of the BEVFusion spray pilot. One end-to-end GPU run of the **pinned**
MIT BEVFusion on Kaggle T4:

1. build the pinned environment (Python 3.8 + torch 1.10.1+cu113 + mmcv-full 1.4.0 + mmdet 2.20.0,
   mmdet3d CUDA ops compiled from the `da93b27` tree);
2. build the detection model from the official config chain;
3. download the official fused checkpoint **inside the kernel** and load it;
4. report **missing / unexpected checkpoint keys**;
5. forward **one sample** and check every output is finite;
6. check the **depth tensor channel count** against what `DepthLSSTransform.dtransform` accepts.

## How to run

Cells are independent: after the bootstrap cell every cell can be re-run on its
own from the Kaggle UI (that is the reason for the cell-per-concern split — the
CLI can only run the whole kernel). Order:

| cell | concern | needs |
|---|---|---|
| bootstrap | writes the helper modules into `/kaggle/working/smoke_test/scripts` | – |
| system probe | GPU, driver, CUDA toolkits, gcc, dataset mount | bootstrap |
| venv 3.8 | `uv python install 3.8.20` + `uv venv` | bootstrap |
| pinned wheels | torch / mmcv-full / mmdet / numpy | venv |
| clone pin | `git clone` + `git checkout da93b27` + static evidence | – |
| cuda toolchain | pick a CUDA 11.x `nvcc` for the extension build | system probe |
| build ops | `python setup.py develop` + per-extension import + GPU micro-smoke | venv, clone, toolchain |
| checkpoint | download + sha256 | – |
| config | resolve the `${...}` config chain | clone |
| model | build BEVFusion | env, ops, config |
| checkpoint load | missing/unexpected keys | model, checkpoint |
| batch | input shapes (synthetic, config-shaped) | config |
| forward | finite check + depth-channel check | model, checkpoint, batch |
| report | aggregate pass/fail + artifacts | the above |

## Environment decisions (recorded, with reasons)

* **Pin `da93b27`**, not HEAD. At HEAD `BaseDepthTransform` grew `add_depth_features`
  / `height_expand` and feeds a 6-channel depth tensor into a `Conv2d(1, 8, 1)`.
  The release commit has neither flag (this notebook greps the clone to prove it).
* **Python 3.8.20 via `uv`**, because Kaggle's system Python is 3.13 and can never
  install torch 1.10.1 / mmcv-full 1.4.0 wheels (they are cp38 only).
* **torch 1.10.1+cu113** from `download.pytorch.org/whl/cu113` and
  **mmcv-full 1.4.0** from `download.openmmlab.com/mmcv/dist/cu113/torch1.10.0`,
  i.e. the exact pinned stack. Nothing is upgraded to make a newer CUDA work.
* **`nvcc` 11.x** is required: torch 1.10.1 raises when the compiling CUDA major
  differs from its own (`torch/utils/cpp_extension.py::_check_cuda_version`).
  Kaggle ships CUDA 12.x only, so the notebook pulls the CUDA 11.8 `nvcc` from
  NVIDIA's `ubuntu2204` apt repository (same apt key as the pre-configured
  `ubuntu2404` repo) and points `CUDA_HOME` at it.
* **torchpack is not installed.** The official entry points use it only to merge
  the config chain and interpolate `${...}`; `smoke_cfg.py` reproduces exactly
  that (root-first `default.yaml` merge, then eval of `${expr}`), because
  torchpack hard-depends on torchvision/tensorpack/tensorboard/scipy.

## Known deviations (all recorded in the artifacts)

* `https://bevfusion.mit.edu/files/pretrained/bevfusion-det.pth` — the URL named in
  the ticket — currently returns **HTTP 404** (the MIT site was rebuilt in 2026).
  The cell probes it first and records the failure, then downloads the same fused
  detection checkpoint from the link the upstream README lists today (Dropbox).
  Which URL served the file, its size and sha256 are recorded.
* The forward uses a **synthetic batch** with the config's shapes: the Kaggle
  mirror has no `nuscenes_infos_*.pkl` yet (open item of ticket 04), so no real
  sample can traverse the MIT pipeline. Nothing here is reported as a real-data
  result; `data_source = "synthetic"` is written into every artifact.

## Acceptance criteria (ticket 02)

- [ ] checkpoint loads; missing/unexpected keys are listed
- [ ] forward of one sample raises no shape error
- [ ] no NaN/Inf in any output
- [ ] depth tensor has exactly the channel count `dtransform` accepts
- [ ] results are saved into a report


In [ ]:
"""Bootstrap (run once per session, idempotent).

Writes the helper modules used by every other cell into
/kaggle/working/smoke_test/scripts and creates the working directories.

Nothing here installs or downloads anything.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
SCRIPTS = os.path.join(WORK, "scripts")
for _dir in (WORK, SCRIPTS, os.path.join(WORK, "artifacts"), os.path.join(WORK, "logs")):
    os.makedirs(_dir, exist_ok=True)

MODULES = {
    "smoke_common.py": r'''"""Shared paths and helpers for the BEVFusion smoke test (ticket 02).

This module runs under the notebook's *system* Python (3.13). It never imports
torch: all pinned work happens in ``/root/bevfusion_smoke/env/bin/python``,
which is a uv-managed CPython 3.8.20 virtualenv.

Every cell of the notebook starts with the same three-line bootstrap, so cells
can be re-run individually from the Kaggle UI:

    import os, sys
    WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
    sys.path.insert(0, WORK)
    import smoke_common as sc

Cell "bootstrap" writes this file, so run it once per session before any other
cell.
"""
import hashlib
import json
import os
import subprocess
import sys
import time

# ---------------------------------------------------------------- pinned facts
PIN_SHORT = "da93b27"
REPO_URL = "https://github.com/mit-han-lab/bevfusion.git"
CONFIG_REL = (
    "configs/nuscenes/det/transfusion/secfpn/camera+lidar/swint_v0p075/convfuser.yaml"
)
CKPT_URL_PRIMARY = "https://bevfusion.mit.edu/files/pretrained/bevfusion-det.pth"
# The primary URL is dead (HTTP 301 -> 404, the MIT site was rebuilt in 2026).
# This is the link the upstream README lists today for the same fused det
# checkpoint; the notebook records which URL actually served the file.
CKPT_URL_FALLBACK = (
    "https://www.dropbox.com/scl/fi/ulaz9z4wdwtypjhx7xdi3/bevfusion-det.pth"
    "?rlkey=ovusfi2rchjub5oafogou255v&dl=1"
)
DATASET_ROOT_CANDIDATES = (
    "/kaggle/input/datasets/sahangunasekara92/nuscenes-v1-0-full-keyframes",
    "/kaggle/input/nuscenes-v1-0-full-keyframes",
)

# ---------------------------------------------------------------------- layout
WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
# Big, disposable things (venv, clone, checkpoint, torch caches) live outside
# the notebook output directory so they are not uploaded with the version.
DEFAULT_ROOT = os.environ.get("SMOKE_ROOT", "/root/bevfusion_smoke")


def _writable(path):
    try:
        os.makedirs(path, exist_ok=True)
        probe = os.path.join(path, ".write_probe")
        with open(probe, "w") as handle:
            handle.write("1")
        os.remove(probe)
        return True
    except Exception:  # noqa: BLE001 - any failure means "not writable"
        return False


os.makedirs(WORK, exist_ok=True)
ROOT = DEFAULT_ROOT
if not _writable(ROOT):
    ROOT = os.path.join(WORK, "root")
    print(
        "[smoke_common] %s is not writable, keeping scratch inside %s instead"
        % (DEFAULT_ROOT, ROOT)
    )
    os.makedirs(ROOT, exist_ok=True)

ART = os.path.join(WORK, "artifacts")
LOGS = os.path.join(WORK, "logs")
SCRIPTS = os.path.join(WORK, "scripts")
ENV = os.path.join(ROOT, "env")
PY = os.path.join(ENV, "bin", "python")
REPO = os.path.join(ROOT, "bevfusion")
CONFIG_PATH = os.path.join(REPO, CONFIG_REL)
CKPT_DIR = os.path.join(ROOT, "ckpt")
CKPT = os.path.join(CKPT_DIR, "bevfusion-det.pth")
TORCH_HOME = os.path.join(ROOT, "torch_home")
TORCH_EXT = os.path.join(ROOT, "torch_extensions")
STATE_PATH = os.path.join(WORK, "state.json")

for _d in (WORK, ART, LOGS, SCRIPTS, ROOT, CKPT_DIR, TORCH_HOME, TORCH_EXT):
    os.makedirs(_d, exist_ok=True)

os.environ["SMOKE_WORK"] = WORK
os.environ["SMOKE_ROOT"] = ROOT


# ----------------------------------------------------------------------- state
def state():
    if not os.path.isfile(STATE_PATH):
        return {}
    with open(STATE_PATH) as f:
        return json.load(f)


def update(**kw):
    current = state()
    current.update(kw)
    with open(STATE_PATH, "w") as f:
        json.dump(current, f, indent=2, sort_keys=True, default=_json_default)
    return current


def _json_default(obj):
    for attr in ("tolist", "item"):
        if hasattr(obj, attr):
            try:
                return getattr(obj, attr)()
            except Exception:
                pass
    return repr(obj)


def save_json(name, obj):
    path = os.path.join(ART, name)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, sort_keys=True, default=_json_default)
    print("wrote " + path)
    return path


def load_json(name):
    path = os.path.join(ART, name)
    if not os.path.isfile(path):
        return {}
    with open(path) as f:
        return json.load(f)


# --------------------------------------------------------------------- command
def sh(cmd, log=None, env=None, cwd=None, timeout=None, echo=True):
    """Run a shell command, tee stdout+stderr into ``log``, return (rc, text)."""
    if isinstance(cmd, (list, tuple)):
        cmd = " ".join(cmd)
    print("$ " + cmd, flush=True)
    started = time.time()
    handle = open(log, "a") if log else None
    if handle:
        handle.write("\n$ " + cmd + "\n")
        handle.flush()
    proc = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    lines = []
    try:
        for line in proc.stdout:
            line = line.rstrip("\n")
            lines.append(line)
            if len(lines) > 6000:
                del lines[:3000]
            if handle:
                handle.write(line + "\n")
                handle.flush()
            if echo:
                print(line, flush=True)
    finally:
        proc.wait(timeout=timeout)
        if handle:
            handle.close()
    print("[exit=%s in %.1fs]" % (proc.returncode, time.time() - started), flush=True)
    return proc.returncode, "\n".join(lines)


def capture(cmd, env=None, cwd=None):
    """Run a command and return its combined output (never raises)."""
    try:
        done = subprocess.run(
            cmd,
            shell=True,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            timeout=600,
        )
        return done.stdout.strip()
    except Exception as exc:  # noqa: BLE001 - diagnostic helper
        return "ERROR: %r" % (exc,)


def sha256(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


# -------------------------------------------------------------------- modules
def write_script(name, source):
    path = os.path.join(SCRIPTS, name)
    with open(path, "w") as handle:
        handle.write(source)
    return path


def py_env(**extra):
    env = dict(os.environ)
    env.update(
        {
            "PYTHONUNBUFFERED": "1",
            "TORCH_HOME": TORCH_HOME,
            "TORCH_EXTENSIONS_DIR": TORCH_EXT,
            "ART": ART,
            "WORK": WORK,
            "SCRIPTS": SCRIPTS,
            "REPO": REPO,
            "CONFIG_PATH": CONFIG_PATH,
            "CKPT_PATH": CKPT,
            "SMOKE_STATE": STATE_PATH,
        }
    )
    env.update(extra)
    return env


def venv_ok():
    if not os.path.isfile(PY):
        return False, "missing " + PY
    out = capture('"%s" -c "import sys; print(sys.version.split()[0])"' % PY)
    return out.startswith("3.8"), out


def run_script(name, log_name=None, env=None, timeout=None, echo=True):
    log = os.path.join(LOGS, log_name or (name + ".log"))
    script = os.path.join(SCRIPTS, name)
    return sh('"%s" "%s"' % (PY, script), log=log, env=env or py_env(), timeout=timeout, echo=echo)


def dataset_root():
    for candidate in DATASET_ROOT_CANDIDATES:
        if os.path.isdir(os.path.join(candidate, "v1.0-trainval")):
            return candidate
    for candidate in DATASET_ROOT_CANDIDATES:
        if os.path.isdir(candidate):
            return candidate
    return None
''',
    "smoke_cfg.py": r'''"""Resolve BEVFusion (da93b27) configs exactly like the official entry points.

`tools/test.py` does::

    configs.load(args.config, recursive=True)      # torchpack
    cfg = Config(recursive_eval(configs), ...)     # mmcv + mmdet3d.utils

torchpack's ``Config.load(recursive=True)`` merges ``default.yaml`` from every
ancestor directory of the leaf config, root first, then applies the leaf file.
mmdet3d's ``recursive_eval`` then evaluates every ``${...}`` string with the
config itself as the eval namespace.

torchpack is *not* installed in the pinned env on purpose: it hard-depends on
torchvision, tensorpack, tensorboard and scipy, which drags a second framework
stack into a torch 1.10.1 environment. This module reproduces the two loaders
above with the same semantics (verified by the invariants printed in the
"config" cell) and nothing else.

Run as a script to dump the resolved config and a config snapshot:

    python smoke_cfg.py
"""

import hashlib
import json
import os
import sys

import yaml


class DotDict(dict):
    """dict with attribute access, like torchpack's ``Config``."""

    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(name)


def chain(fpath):
    """Files torchpack would merge, in application order (root first)."""
    fpath = os.path.abspath(fpath)
    ext = os.path.splitext(fpath)[1]
    candidates = [fpath]
    while os.path.dirname(fpath) != fpath:
        fpath = os.path.dirname(fpath)
        candidates.append(os.path.join(fpath, "default" + ext))
    return [path for path in reversed(candidates) if os.path.exists(path)]


def _merge(dst, src):
    for key, value in src.items():
        if isinstance(value, dict):
            if not isinstance(dst.get(key), dict):
                dst[key] = {}
            _merge(dst[key], value)
        else:
            dst[key] = value
    return dst


def _wrap(obj):
    if isinstance(obj, dict):
        return DotDict((key, _wrap(value)) for key, value in obj.items())
    if isinstance(obj, list):
        return [_wrap(value) for value in obj]
    return obj


def _plain(obj):
    if isinstance(obj, dict):
        return dict((key, _plain(value)) for key, value in obj.items())
    if isinstance(obj, (list, tuple)):
        return [_plain(value) for value in obj]
    return obj


def _resolve(obj, namespace, depth=0):
    if depth > 32:
        raise RuntimeError("config interpolation nested too deeply")
    if isinstance(obj, dict):
        return dict((key, _resolve(value, namespace, depth + 1)) for key, value in obj.items())
    if isinstance(obj, list):
        return [_resolve(value, namespace, depth + 1) for value in obj]
    if isinstance(obj, str):
        text = obj.strip()
        if text.startswith("${") and text.endswith("}"):
            # torchpack/mmdet3d eval the expression with the config as globals.
            value = eval(text[2:-1], {"__builtins__": {}}, namespace)  # noqa: S307
            return _resolve(value, namespace, depth + 1)
    return obj


def load(fpath):
    """Return (resolved plain dict, list of config files that contributed)."""
    merged = {}
    files = chain(fpath)
    for path in files:
        with open(path) as handle:
            data = yaml.safe_load(handle) or {}
        _merge(merged, data)
    namespace = _wrap(merged)
    return _plain(_resolve(merged, namespace)), files


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        digest.update(handle.read())
    return digest.hexdigest()


def _unresolved(obj):
    if isinstance(obj, dict):
        for value in obj.values():
            found = _unresolved(value)
            if found:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = _unresolved(value)
            if found:
                return found
    elif isinstance(obj, str) and "${" in obj:
        return obj
    return None


def check(cfg):
    """Invariants that the smoke test relies on. Returns a dict of results."""
    vt = cfg["model"]["encoders"]["camera"]["vtransform"]
    head = cfg["model"]["heads"]["object"]
    checks = {
        "no_unresolved_interpolation": _unresolved(cfg) is None,
        "unresolved_example": _unresolved(cfg),
        "image_size": cfg["image_size"],
        "point_cloud_range": cfg["point_cloud_range"],
        "voxel_size": cfg["voxel_size"],
        "vtransform_type": vt["type"],
        "vtransform_in_channels": vt["in_channels"],
        "vtransform_out_channels": vt["out_channels"],
        "vtransform_feature_size": vt["feature_size"],
        "vtransform_dbound": vt["dbound"],
        "vtransform_downsample": vt["downsample"],
        "vtransform_image_size": vt["image_size"],
        "fuser_type": cfg["model"]["fuser"]["type"],
        "fuser_in_channels": cfg["model"]["fuser"]["in_channels"],
        "fuser_out_channels": cfg["model"]["fuser"]["out_channels"],
        "lidar_voxel_size": cfg["model"]["encoders"]["lidar"]["voxelize"]["voxel_size"],
        "lidar_sparse_shape": cfg["model"]["encoders"]["lidar"]["backbone"]["sparse_shape"],
        "lidar_output_channels": cfg["model"]["encoders"]["lidar"]["backbone"]["output_channels"],
        "head_test_voxel_size": head["test_cfg"]["voxel_size"],
        "head_test_grid_size": head["test_cfg"]["grid_size"],
        "head_num_classes": head["num_classes"],
        "camera_backbone_type": cfg["model"]["encoders"]["camera"]["backbone"]["type"],
    }
    # The release commit has neither of the depth flags that HEAD grew later.
    dumped = json.dumps(cfg)
    checks["add_depth_features_present"] = "add_depth_features" in dumped
    checks["height_expand_present"] = "height_expand" in dumped
    checks["expected"] = {
        "image_size": [256, 704],
        "point_cloud_range": [-54.0, -54.0, -5.0, 54.0, 54.0, 3.0],
        "voxel_size": [0.075, 0.075, 0.2],
        "vtransform_type": "DepthLSSTransform",
        "vtransform_out_channels": 80,
        "fuser_in_channels": [80, 256],
        "no_unresolved_interpolation": True,
        "add_depth_features_present": False,
        "height_expand_present": False,
    }
    passed = []
    failed = []
    for key, expected in checks["expected"].items():
        (passed if checks.get(key) == expected else failed).append(key)
    checks["passed"] = passed
    checks["failed"] = failed
    checks["ok"] = not failed
    return checks


def main():
    path = os.environ.get("CONFIG_PATH") or sys.argv[1]
    art = os.environ.get("ART") or os.getcwd()
    if not os.path.isdir(art):
        os.makedirs(art)
    cfg, files = load(path)
    snapshot = {
        "leaf": os.path.abspath(path),
        "chain": [{"path": p, "sha256": sha256(p)} for p in files],
        "config_sha256_of_chain": hashlib.sha256(
            json.dumps([sha256(p) for p in files]).encode()
        ).hexdigest(),
    }
    checks = check(cfg)
    with open(os.path.join(art, "cfg_resolved.json"), "w") as handle:
        json.dump(cfg, handle, indent=2, sort_keys=True)
    with open(os.path.join(art, "config_snapshot.json"), "w") as handle:
        json.dump(snapshot, handle, indent=2, sort_keys=True)
    with open(os.path.join(art, "cfg_checks.json"), "w") as handle:
        json.dump(checks, handle, indent=2, sort_keys=True)
    print(json.dumps(snapshot, indent=2))
    print("resolved config keys: " + ", ".join(sorted(cfg.keys())))
    print("model section:")
    print(json.dumps(cfg["model"], indent=2, sort_keys=True)[:8000])
    print("checks:")
    print(json.dumps(checks, indent=2, sort_keys=True))
    print("CONFIG_OK" if checks["ok"] else "CONFIG_FAILED")


if __name__ == "__main__":
    main()
''',
    "smoke_batch.py": r'''"""Build the batch that is fed to the pinned model.

Real nuScenes keyframes cannot be used yet: ``nuscenes_infos_*.pkl`` do not
exist for the Kaggle mirror (open item of ticket 04), so this builds a
**synthetic** batch whose shapes and value ranges follow the resolved config
exactly:

    img                [B, 6, 3, 256, 704]
    points             list of B float32 [N, 5] inside point_cloud_range
    camera2ego         [B, 6, 4, 4]
    lidar2ego          [B, 4, 4]
    lidar2camera       [B, 6, 4, 4]   = inv(camera2ego) @ lidar2ego
    lidar2image        [B, 6, 4, 4]   = intrinsics @ lidar2camera
    camera_intrinsics  [B, 6, 4, 4]
    img_aug_matrix     [B, 6, 4, 4]   identity (test-time ImageAug3D in the config)
    lidar_aug_matrix   [B, 4, 4]      identity (test-time GlobalRotScaleTrans)

The geometry is internally consistent (real pinhole intrinsics, cameras placed
around the ego origin), so the LiDAR points really do project into the images
and the depth tensor is not trivially zero. It is still synthetic data: every
downstream artifact records ``data_source = "synthetic"``.
"""

import json
import math
import os

import torch

IMAGE_H = 256
IMAGE_W = 704
NUM_CAMS = 6
NUM_POINTS = 20000
POINT_CLOUD_RANGE = (-54.0, -54.0, -5.0, 54.0, 54.0, 3.0)

CAM_ORDER = (
    "CAM_FRONT",
    "CAM_FRONT_LEFT",
    "CAM_FRONT_RIGHT",
    "CAM_BACK",
    "CAM_BACK_LEFT",
    "CAM_BACK_RIGHT",
)

# Rough nuScenes layout: yaw around the ego z axis, in radians.
CAM_YAW = {
    "CAM_FRONT": 0.0,
    "CAM_FRONT_LEFT": 0.9553166181,
    "CAM_FRONT_RIGHT": -0.9553166181,
    "CAM_BACK": math.pi,
    "CAM_BACK_LEFT": -2.1862760355,
    "CAM_BACK_RIGHT": 2.1862760355,
}


def _rot_z(angle):
    cos, sin = math.cos(angle), math.sin(angle)
    return torch.tensor(
        [[cos, -sin, 0.0], [sin, cos, 0.0], [0.0, 0.0, 1.0]], dtype=torch.float64
    )


def camera2ego(yaw, translation=(0.5, 0.0, 1.6)):
    """Camera frame (x right, y down, z forward) to ego (x fwd, y left, z up)."""
    base = torch.tensor(
        [[0.0, 0.0, 1.0], [-1.0, 0.0, 0.0], [0.0, -1.0, 0.0]], dtype=torch.float64
    )
    mat = torch.eye(4, dtype=torch.float64)
    mat[:3, :3] = _rot_z(yaw) @ base
    mat[:3, 3] = torch.tensor(translation, dtype=torch.float64)
    return mat


def intrinsics(fx=400.0, fy=400.0, cx=IMAGE_W / 2.0, cy=IMAGE_H / 2.0):
    mat = torch.eye(4, dtype=torch.float64)
    mat[0, 0], mat[1, 1] = fx, fy
    mat[0, 2], mat[1, 2] = cx, cy
    return mat


def build(batch_size=1, num_points=NUM_POINTS, seed=0, device="cuda"):
    """Return (batch dict, description dict)."""
    generator = torch.Generator().manual_seed(seed)
    xmin, ymin, zmin, xmax, ymax, zmax = POINT_CLOUD_RANGE

    points = []
    for _ in range(batch_size):
        xyz = torch.stack(
            [
                torch.rand(num_points, generator=generator) * (xmax - xmin) + xmin,
                torch.rand(num_points, generator=generator) * (ymax - ymin) + ymin,
                torch.rand(num_points, generator=generator) * (zmax - zmin) + zmin,
            ],
            dim=1,
        )
        intensity = torch.rand(num_points, 1, generator=generator)
        ring = torch.randint(0, 32, (num_points, 1), generator=generator).float()
        points.append(torch.cat([xyz, intensity, ring], dim=1).float())

    img = torch.randn(batch_size, NUM_CAMS, 3, IMAGE_H, IMAGE_W, generator=generator)

    cam2ego = torch.stack(
        [camera2ego(CAM_YAW[name]) for name in CAM_ORDER]
    )  # [6, 4, 4]
    cam2ego = cam2ego.unsqueeze(0).repeat(batch_size, 1, 1, 1)
    lidar2ego = torch.eye(4, dtype=torch.float64).unsqueeze(0).repeat(batch_size, 1, 1)
    lidar2camera = torch.matmul(cam2ego.inverse(), lidar2ego.unsqueeze(1))
    cam_intr = intrinsics().unsqueeze(0).repeat(NUM_CAMS, 1, 1)
    cam_intr = cam_intr.unsqueeze(0).repeat(batch_size, 1, 1, 1)
    lidar2image = torch.matmul(cam_intr, lidar2camera)
    img_aug = torch.eye(4, dtype=torch.float64).unsqueeze(0).repeat(batch_size, NUM_CAMS, 1, 1)
    lidar_aug = torch.eye(4, dtype=torch.float64).unsqueeze(0).repeat(batch_size, 1, 1)

    batch = {
        "img": img.float().to(device),
        "points": [p.to(device) for p in points],
        "camera2ego": cam2ego.float().to(device),
        "lidar2ego": lidar2ego.float().to(device),
        "lidar2camera": lidar2camera.float().to(device),
        "lidar2image": lidar2image.float().to(device),
        "camera_intrinsics": cam_intr.float().to(device),
        "img_aug_matrix": img_aug.float().to(device),
        "lidar_aug_matrix": lidar_aug.float().to(device),
    }
    description = {
        "data_source": "synthetic",
        "why_synthetic": (
            "nuscenes_infos_{train,val}.pkl are missing on the Kaggle mirror, "
            "so no real sample can go through the MIT pipeline yet (ticket 04)"
        ),
        "batch_size": batch_size,
        "num_points_per_sample": num_points,
        "seed": seed,
        "device": str(device),
        "shapes": dict((key, _shape(value)) for key, value in batch.items()),
        "point_cloud_range": list(POINT_CLOUD_RANGE),
        "cam_order": list(CAM_ORDER),
        "intrinsics_4x4": intrinsics().tolist(),
        "camera2ego_CAM_FRONT": camera2ego(CAM_YAW["CAM_FRONT"]).tolist(),
    }
    return batch, description


def _shape(value):
    if torch.is_tensor(value):
        return list(value.shape)
    if isinstance(value, list) and value and torch.is_tensor(value[0]):
        return [len(value)] + list(value[0].shape)
    return None


if __name__ == "__main__":
    batch, description = build(batch_size=1, device="cpu")
    print(json.dumps(description, indent=2))
    for key in sorted(batch):
        value = batch[key]
        if torch.is_tensor(value):
            print(
                "%-18s %-22s finite=%s min=%.4f max=%.4f"
                % (
                    key,
                    str(tuple(value.shape)),
                    bool(torch.isfinite(value).all()),
                    float(value.min()),
                    float(value.max()),
                )
            )
        else:
            tensor = value[0]
            print(
                "%-18s %-22s finite=%s min=%.4f max=%.4f"
                % (
                    key,
                    str([len(value)] + list(tensor.shape)),
                    bool(torch.isfinite(tensor).all()),
                    float(tensor.min()),
                    float(tensor.max()),
                )
            )
    art = os.environ.get("ART")
    if art:
        with open(os.path.join(art, "batch.json"), "w") as handle:
            json.dump(description, handle, indent=2, sort_keys=True)
    print("BATCH_OK")
''',
    "smoke_model.py": r'''"""Build the pinned BEVFusion model, load the official fused checkpoint, and
summarise tensors. Everything here runs inside the pinned Python 3.8 env
(torch 1.10.1+cu113, mmcv-full 1.4.0, mmdet 2.20.0, mmdet3d built from the
da93b27 source tree).
"""

import copy
import hashlib
import json
import os
import time

import torch


# --------------------------------------------------------------------- tensors
def tensor_stats(tensor, max_elems=4000000):
    """Small, JSON-friendly summary of a tensor, including a finiteness check."""
    if tensor is None or not torch.is_tensor(tensor):
        return None
    stats = {
        "shape": list(tensor.shape),
        "dtype": str(tensor.dtype),
        "numel": int(tensor.numel()),
    }
    flat = tensor.detach().reshape(-1)
    sample = flat[:max_elems] if flat.numel() > max_elems else flat
    floats = sample.float()
    stats["finite_sample"] = bool(torch.isfinite(floats).all())
    stats["samples_checked"] = int(sample.numel())
    if sample.numel():
        finite = floats[torch.isfinite(floats)]
        if finite.numel():
            stats["min"] = float(finite.min())
            stats["max"] = float(finite.max())
            stats["abs_max"] = float(finite.abs().max())
            stats["mean"] = float(finite.mean())
        stats["nan_count_sample"] = int(torch.isnan(floats).sum())
        stats["inf_count_sample"] = int(torch.isinf(floats).sum())
        stats["nonzero_sample"] = int((floats != 0).sum())
    return stats


def finite_ok(stats):
    if stats is None:
        return True
    return bool(stats.get("finite_sample")) and int(stats.get("nan_count_sample", 0)) == 0


def _sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


# ----------------------------------------------------------------------- model
def build(cfg_model, device="cuda", allow_init_cfg_fallback=True):
    """Build BEVFusion from the resolved ``cfg['model']`` dict."""
    from mmdet3d.models import build_model

    info = {"init_cfg_disabled": False, "init_cfg_error": None, "build_seconds": None}
    started = time.time()
    try:
        model = build_model(copy.deepcopy(cfg_model))
    except Exception as exc:  # noqa: BLE001 - reported verbatim
        info["init_cfg_error"] = repr(exc)[:4000]
        if not allow_init_cfg_fallback:
            raise
        # The Swin backbone initialises from a GitHub-hosted Swin-Tiny
        # checkpoint (model.encoders.camera.backbone.init_cfg). If that download
        # fails, build without it: every weight is overwritten by the fused
        # checkpoint we load next, so the smoke test is unaffected. The flag is
        # reported so nobody mistakes it for the pristine config.
        fallback = copy.deepcopy(cfg_model)
        fallback["encoders"]["camera"]["backbone"]["init_cfg"] = None
        model = build_model(fallback)
        info["init_cfg_disabled"] = True
    model = model.to(device)
    model.eval()
    info["build_seconds"] = round(time.time() - started, 2)
    info["num_params"] = int(sum(p.numel() for p in model.parameters()))
    info["num_buffers"] = int(sum(b.numel() for b in model.buffers()))
    info["num_state_dict_keys"] = len(model.state_dict())
    info["modules"] = dict(
        (name, type(module).__name__) for name, module in model.named_children()
    )
    info["dtype"] = str(next(model.parameters()).dtype)
    info["device"] = str(next(model.parameters()).device)
    return model, info


# ------------------------------------------------------------------ checkpoint
def load_checkpoint(model, path, device="cuda"):
    """Load a checkpoint the way mmcv does, but report every mismatch."""
    report = {
        "path": path,
        "size_bytes": os.path.getsize(path),
        "sha256": _sha256(path),
    }
    started = time.time()
    ckpt = torch.load(path, map_location="cpu")
    report["checkpoint_object_type"] = type(ckpt).__name__
    report["top_level_keys"] = sorted(ckpt.keys()) if isinstance(ckpt, dict) else []
    report["meta"] = _jsonable(ckpt.get("meta")) if isinstance(ckpt, dict) else None
    state = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    state = dict(state)
    report["ckpt_num_keys"] = len(state)
    module_prefixed = [key for key in state if key.startswith("module.")]
    report["module_prefixed_keys"] = len(module_prefixed)
    if module_prefixed and len(module_prefixed) == len(state):
        state = dict((key[len("module."):], value) for key, value in state.items())
        report["module_prefix_stripped"] = True
    else:
        report["module_prefix_stripped"] = False

    own = model.state_dict()
    report["model_num_keys"] = len(own)
    missing = sorted(set(own) - set(state))
    unexpected = sorted(set(state) - set(own))
    shape_mismatch = []
    for key in sorted(set(own) & set(state)):
        model_shape = tuple(own[key].shape)
        ckpt_shape = tuple(state[key].shape)
        if model_shape != ckpt_shape:
            shape_mismatch.append(
                {"key": key, "model_shape": list(model_shape), "ckpt_shape": list(ckpt_shape)}
            )
    report["shape_mismatch"] = shape_mismatch[:200]
    report["shape_mismatch_count"] = len(shape_mismatch)

    load_error = None
    try:
        result = model.load_state_dict(state, strict=False)
        missing = sorted(set(missing) | set(result.missing_keys))
        unexpected = sorted(set(unexpected) | set(result.unexpected_keys))
    except Exception as exc:  # noqa: BLE001 - a shape mismatch raises here
        load_error = repr(exc)[:4000]
    report["load_error"] = load_error

    # Missing keys that actually carry weights (num_batches_tracked does not).
    weight_affecting = []
    for key in missing:
        if key not in own:
            continue
        if key.endswith("num_batches_tracked"):
            continue
        if own[key].numel() > 0:
            weight_affecting.append(key)
    report["missing_keys"] = missing[:300]
    report["missing_keys_total"] = len(missing)
    report["missing_keys_weight_affecting"] = weight_affecting[:300]
    report["missing_keys_weight_affecting_total"] = len(weight_affecting)
    report["unexpected_keys"] = unexpected[:300]
    report["unexpected_keys_total"] = len(unexpected)
    report["matched_keys"] = len(set(own) & set(state)) - len(shape_mismatch)
    report["load_seconds"] = round(time.time() - started, 2)

    # Direct value check: proves the checkpoint really landed in the module.
    samples = []
    mismatched = set(item["key"] for item in shape_mismatch)
    for key in sorted(set(own) & set(state))[:5]:
        if key in mismatched:
            continue
        try:
            model_value = model.state_dict()[key].detach().cpu().float()
            ckpt_value = state[key].detach().cpu().float()
            samples.append(
                {"key": key, "identical": bool(torch.equal(model_value, ckpt_value))}
            )
        except Exception as exc:  # noqa: BLE001
            samples.append({"key": key, "identical": False, "error": repr(exc)[:300]})
    report["weight_sample_check"] = samples
    report["ok"] = bool(
        load_error is None
        and not shape_mismatch
        and not weight_affecting
        and all(item["identical"] for item in samples)
    )
    return report


def cross_check_mmcv(model, path):
    """Official loader (mmcv.runner.load_checkpoint) as a second opinion."""
    from mmcv.runner import load_checkpoint as mmcv_load_checkpoint

    result = {"used": True}
    try:
        mmcv_load_checkpoint(model, path, map_location="cpu", strict=False)
        result["ok"] = True
    except Exception as exc:  # noqa: BLE001
        result["ok"] = False
        result["error"] = repr(exc)[:4000]
    return result


def _jsonable(obj):
    try:
        json.dumps(obj)
        return obj
    except TypeError:
        return repr(obj)


def dump(path, payload):
    with open(path, "w") as handle:
        json.dump(payload, handle, indent=2, sort_keys=True, default=_jsonable)
''',
    "step_env.py": r'''"""Verify the pinned stack inside the uv-managed Python 3.8 venv."""

import json
import os
import sys

report = {"python": sys.version, "python_version": list(sys.version_info[:3]),
          "executable": sys.executable}

try:
    import numpy

    report["numpy"] = numpy.__version__
except Exception as exc:  # noqa: BLE001
    report["numpy_error"] = repr(exc)

import torch

report["torch"] = torch.__version__
report["torch_cuda_build"] = torch.version.cuda
report["cudnn"] = torch.backends.cudnn.version()
report["cuda_available"] = bool(torch.cuda.is_available())
if torch.cuda.is_available():
    report["device_count"] = torch.cuda.device_count()
    report["device_0"] = torch.cuda.get_device_name(0)
    report["device_0_capability"] = list(torch.cuda.get_device_capability(0))
    report["device_0_total_memory_gb"] = round(
        torch.cuda.get_device_properties(0).total_memory / float(2 ** 30), 2
    )

import mmcv

report["mmcv"] = mmcv.__version__

import mmdet

report["mmdet"] = mmdet.__version__

report["pinned_ok"] = bool(
    report["torch"].startswith("1.10.1")
    and report.get("torch_cuda_build") == "11.3"
    and report.get("mmcv") == "1.4.0"
    and report.get("mmdet") == "2.20.0"
    and report["python_version"][:2] == [3, 8]
    and report["cuda_available"]
)

print(json.dumps(report, indent=2, sort_keys=True))
art = os.environ.get("ART")
if art:
    with open(os.path.join(art, "env.json"), "w") as handle:
        json.dump(report, handle, indent=2, sort_keys=True)
print("ENV_OK" if report["pinned_ok"] else "ENV_PINNED_MISMATCH")
''',
    "step_ops_import.py": r'''"""Check every compiled mmdet3d extension, and run two GPU micro-smokes.

Import success alone does not prove the binaries are usable (an ABI or arch
mismatch only shows up when a kernel launches), so this also runs the two ops
the detection forward needs: ``bev_pool`` and hard voxelization.
"""

import json
import os
import sys
import traceback

EXTENSIONS = [
    "mmdet3d.ops.bev_pool.bev_pool_ext",
    "mmdet3d.ops.voxel.voxel_layer",
    "mmdet3d.ops.iou3d.iou3d_cuda",
    "mmdet3d.ops.roiaware_pool3d.roiaware_pool3d_ext",
    "mmdet3d.ops.ball_query.ball_query_ext",
    "mmdet3d.ops.knn.knn_ext",
    "mmdet3d.ops.paconv.assign_score_withk_ext",
    "mmdet3d.ops.group_points.group_points_ext",
    "mmdet3d.ops.interpolate.interpolate_ext",
    "mmdet3d.ops.furthest_point_sample.furthest_point_sample_ext",
    "mmdet3d.ops.gather_points.gather_points_ext",
    "mmdet3d.ops.spconv.sparse_conv_ext",
]

report = {"extensions": {}, "package_import": None, "gpu_smoke": {}}

try:
    import mmdet3d

    report["mmdet3d_path"] = mmdet3d.__file__
except Exception as exc:  # noqa: BLE001
    report["mmdet3d_path"] = "IMPORT FAILED: " + repr(exc)

for name in EXTENSIONS:
    try:
        __import__(name)
        report["extensions"][name] = {"ok": True}
    except Exception as exc:  # noqa: BLE001
        report["extensions"][name] = {
            "ok": False,
            "error": repr(exc)[:1500],
            "traceback": traceback.format_exc()[-1500:],
        }

try:
    import mmdet3d.ops as ops

    report["package_import"] = {"ok": True}
    report["symbols"] = {
        "bev_pool": hasattr(ops, "bev_pool"),
        "Voxelization": hasattr(ops, "Voxelization"),
        "SparseBasicBlock": hasattr(ops, "SparseBasicBlock"),
    }
except Exception as exc:  # noqa: BLE001
    report["package_import"] = {"ok": False, "error": repr(exc)[:2000]}

if report["package_import"].get("ok"):
    import torch
    from mmdet3d.ops import Voxelization, bev_pool

    device = "cuda"
    # --- bev_pool: 64 frustum cells, 4 channels -> [1, 4, 4, 8, 8]
    # coords columns are (x_idx, y_idx, z_idx, batch_idx), as in
    # mmdet3d/models/vtransforms/base.py::bev_pool.
    try:
        torch.manual_seed(0)
        feats = torch.rand(64, 4, device=device)
        coords = torch.stack(
            [
                torch.randint(0, 8, (64,), device=device),
                torch.randint(0, 8, (64,), device=device),
                torch.randint(0, 4, (64,), device=device),
                torch.zeros(64, device=device, dtype=torch.long),
            ],
            dim=1,
        ).long()
        out = bev_pool(feats, coords, 1, 4, 8, 8)
        report["gpu_smoke"]["bev_pool"] = {
            "ok": True,
            "shape": list(out.shape),
            "finite": bool(torch.isfinite(out).all()),
        }
    except Exception as exc:  # noqa: BLE001
        report["gpu_smoke"]["bev_pool"] = {"ok": False, "error": repr(exc)[:2000]}

    # --- hard voxelization on a random cloud inside the detection range
    try:
        torch.manual_seed(0)
        points = torch.rand(50000, 5, device=device)
        points[:, 0] = points[:, 0] * 108.0 - 54.0
        points[:, 1] = points[:, 1] * 108.0 - 54.0
        points[:, 2] = points[:, 2] * 8.0 - 5.0
        voxelize = Voxelization(
            voxel_size=[0.075, 0.075, 0.2],
            point_cloud_range=[-54.0, -54.0, -5.0, 54.0, 54.0, 3.0],
            max_num_points=10,
            max_voxels=[120000, 160000],
        ).to(device)
        feats, coords, counts = voxelize(points)
        report["gpu_smoke"]["voxelize"] = {
            "ok": True,
            "voxels": int(feats.shape[0]),
            "feat_shape": list(feats.shape),
            "coord_shape": list(coords.shape),
            "counts_shape": list(counts.shape),
        }
    except Exception as exc:  # noqa: BLE001
        report["gpu_smoke"]["voxelize"] = {"ok": False, "error": repr(exc)[:2000]}

report["ok"] = bool(
    report["package_import"].get("ok")
    and all(item.get("ok") for item in report["extensions"].values())
    and all(item.get("ok") for item in report["gpu_smoke"].values())
)

print(json.dumps(report, indent=2, sort_keys=True))
art = os.environ.get("ART")
if art:
    with open(os.path.join(art, "ops.json"), "w") as handle:
        json.dump(report, handle, indent=2, sort_keys=True)
print("OPS_OK" if report["ok"] else "OPS_FAILED")
''',
    "step_model.py": r'''"""Build the pinned BEVFusion detection model and report its structure."""

import json
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

import smoke_cfg
import smoke_model

config_path = os.environ["CONFIG_PATH"]
cfg, files = smoke_cfg.load(config_path)

model, info = smoke_model.build(cfg["model"], device="cuda")

vtransform = model.encoders["camera"]["vtransform"]
dtransform = vtransform.dtransform
dtransform_first = dtransform[0]

groups = {}
for name, module in model.named_children():
    groups[name] = {
        "type": type(module).__name__,
        "params": int(sum(p.numel() for p in module.parameters())),
    }
for name, module in model.encoders["camera"].named_children():
    groups["encoders.camera." + name] = {
        "type": type(module).__name__,
        "params": int(sum(p.numel() for p in module.parameters())),
    }
for name, module in model.encoders["lidar"].named_children():
    groups["encoders.lidar." + name] = {
        "type": type(module).__name__,
        "params": int(sum(p.numel() for p in module.parameters())),
    }
for name, module in model.decoder.named_children():
    groups["decoder." + name] = {
        "type": type(module).__name__,
        "params": int(sum(p.numel() for p in module.parameters())),
    }

payload = {
    "config_path": config_path,
    "config_files": [os.path.relpath(p, os.environ["REPO"]) for p in files],
    "model_params": info["num_params"],
    "model_buffers": info["num_buffers"],
    "state_dict_keys": info["num_state_dict_keys"],
    "build_seconds": info["build_seconds"],
    "init_cfg_disabled": info["init_cfg_disabled"],
    "init_cfg_error": info["init_cfg_error"],
    "groups": groups,
    "vtransform": {
        "type": type(vtransform).__name__,
        "in_channels": vtransform.in_channels,
        "out_channels": vtransform.out_channels,
        "D": int(vtransform.D),
        "image_size": list(vtransform.image_size),
        "feature_size": list(vtransform.feature_size),
        "dbound": list(vtransform.dbound),
        "dtransform_first_layer": str(dtransform_first),
        "dtransform_in_channels": int(dtransform_first.in_channels),
        "camera_count_hardcoded": 6,
    },
}
smoke_model.dump(os.path.join(os.environ["ART"], "model.json"), payload)
print(json.dumps(payload, indent=2, sort_keys=True))
print("MODEL_OK")
''',
    "step_checkpoint.py": r'''"""Load the official fused checkpoint and report every key mismatch."""

import json
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

import torch

import smoke_cfg
import smoke_model

config_path = os.environ["CONFIG_PATH"]
ckpt_path = os.environ["CKPT_PATH"]
art = os.environ["ART"]

cfg, files = smoke_cfg.load(config_path)
model, build_info = smoke_model.build(cfg["model"], device="cuda")

report = smoke_model.load_checkpoint(model, ckpt_path, device="cuda")
report["cross_check_mmcv"] = smoke_model.cross_check_mmcv(model, ckpt_path)

# Areas of interest for the spray pilot: the camera depth path must be fully
# covered by the checkpoint, because Head's depth path is what the pilot touches.
depth_path_keys = [
    key for key in model.state_dict() if "vtransform" in key
]
report["vtransform_param_keys"] = len(depth_path_keys)
report["vtransform_weight_affecting_missing"] = [
    key
    for key in report["missing_keys_weight_affecting"]
    if "vtransform" in key
]

payload = {
    "checkpoint": report,
    "build": {
        "init_cfg_disabled": build_info["init_cfg_disabled"],
        "num_params": build_info["num_params"],
        "state_dict_keys": build_info["num_state_dict_keys"],
    },
}
smoke_model.dump(os.path.join(art, "checkpoint.json"), payload)

print("checkpoint: " + ckpt_path)
print("size_bytes: %d" % report["size_bytes"])
print("sha256: " + report["sha256"])
print("torch object: %s, top level keys: %s" % (report["checkpoint_object_type"], report["top_level_keys"]))
print("ckpt keys: %d, model keys: %d, matched: %d"
      % (report["ckpt_num_keys"], report["model_num_keys"], report["matched_keys"]))
print("shape mismatches: %d" % report["shape_mismatch_count"])
for item in report["shape_mismatch"]:
    print("  SHAPE model=%s ckpt=%s %s" % (item["model_shape"], item["ckpt_shape"], item["key"]))
print("missing keys: %d (weight affecting: %d)"
      % (report["missing_keys_total"], report["missing_keys_weight_affecting_total"]))
for key in report["missing_keys"][:60]:
    print("  MISSING " + key)
print("unexpected keys: %d" % report["unexpected_keys_total"])
for key in report["unexpected_keys"][:60]:
    print("  UNEXPECTED " + key)
print("load_error: %s" % report["load_error"])
print("mmcv cross check: %s" % json.dumps(report["cross_check_mmcv"]))
print("weight sample check: %s" % json.dumps(report["weight_sample_check"]))
print("CKPT_OK" if report["ok"] else "CKPT_FAILED")
''',
    "step_forward.py": r'''"""One forward pass through the pinned model, with finiteness and depth checks.

Evidence produced here:
  * the batch that was fed in (synthetic, shapes match the resolved config);
  * the depth tensor that ``DepthLSSTransform.dtransform`` actually receives
    (channel count, spatial size, non-zero fraction, finiteness);
  * intermediate feature tensors (camera BEV, fused, decoder) and head outputs;
  * the decoded detection result;
  * a pass/fail verdict per acceptance criterion of ticket 02.
"""

import inspect
import json
import os
import sys
import time
import traceback

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

import torch

import smoke_batch
import smoke_cfg
import smoke_model

from mmdet3d.core import LiDARInstance3DBoxes

DEVICE = "cuda"
art = os.environ["ART"]
config_path = os.environ["CONFIG_PATH"]
ckpt_path = os.environ["CKPT_PATH"]


def conv_out(size, kernel, stride, padding):
    return (size + 2 * padding - kernel) // stride + 1


capture = {}
report = {}

# ------------------------------------------------------------------ model+ckpt
cfg, files = smoke_cfg.load(config_path)
model, build_info = smoke_model.build(cfg["model"], device=DEVICE)
ckpt_report = smoke_model.load_checkpoint(model, ckpt_path, device=DEVICE)
report["build"] = {
    "init_cfg_disabled": build_info["init_cfg_disabled"],
    "init_cfg_error": build_info["init_cfg_error"],
    "num_params": build_info["num_params"],
}
report["checkpoint"] = {
    "sha256": ckpt_report["sha256"],
    "size_bytes": ckpt_report["size_bytes"],
    "load_error": ckpt_report["load_error"],
    "shape_mismatch_count": ckpt_report["shape_mismatch_count"],
    "missing_keys_total": ckpt_report["missing_keys_total"],
    "missing_keys_weight_affecting_total": ckpt_report["missing_keys_weight_affecting_total"],
    "unexpected_keys_total": ckpt_report["unexpected_keys_total"],
    "ok": ckpt_report["ok"],
}

# --------------------------------------------------------------------- batch
batch, batch_description = smoke_batch.build(batch_size=1, device=DEVICE)
metas = [
    {
        "box_type_3d": LiDARInstance3DBoxes,
        "img_shape": [(smoke_batch.IMAGE_H, smoke_batch.IMAGE_W)] * smoke_batch.NUM_CAMS,
        "num_views": smoke_batch.NUM_CAMS,
        "scene_token": "synthetic",
        "sample_idx": 0,
    }
    for _ in range(batch_description["batch_size"])
]
report["batch"] = batch_description

# --------------------------------------------------------------------- hooks
vtransform = model.encoders["camera"]["vtransform"]
dtransform = vtransform.dtransform
hooks = []


def depth_pre_hook(module, inputs):
    tensor = inputs[0]
    capture["depth"] = smoke_model.tensor_stats(tensor)
    if tensor.numel():
        capture["depth"]["nonzero_fraction"] = float((tensor != 0).float().mean())


hooks.append(dtransform.register_forward_pre_hook(depth_pre_hook))


def make_post_hook(name):
    def post_hook(module, inputs, outputs):
        tensor = outputs
        if isinstance(outputs, (list, tuple)):
            tensor = outputs[0] if outputs else None
        capture[name] = smoke_model.tensor_stats(tensor)

    return post_hook


hooks.append(vtransform.register_forward_hook(make_post_hook("camera_bev")))
hooks.append(model.fuser.register_forward_hook(make_post_hook("fused_bev")))
hooks.append(model.decoder["neck"].register_forward_hook(make_post_hook("decoder_neck")))


def head_hook(module, inputs, outputs):
    def walk(obj, prefix, depth=0):
        if depth > 3:
            return
        if torch.is_tensor(obj):
            capture[prefix] = smoke_model.tensor_stats(obj)
        elif isinstance(obj, dict):
            for key, value in obj.items():
                walk(value, prefix + "." + str(key), depth + 1)
        elif isinstance(obj, (list, tuple)):
            for index, value in enumerate(obj[:2]):
                walk(value, prefix + "[%d]" % index, depth + 1)

    walk(outputs, "head")


hooks.append(model.heads["object"].register_forward_hook(head_hook))

# -------------------------------------------------------------------- forward
forward_inputs = dict(batch)
forward_inputs["metas"] = metas
forward_error = None
full_forward_ok = False
trunk_only = False
started = time.time()
torch.cuda.reset_peak_memory_stats()
try:
    with torch.no_grad():
        result = model(**forward_inputs)
    torch.cuda.synchronize()
    full_forward_ok = True
except Exception as exc:  # noqa: BLE001 - reported verbatim, then trunk-only retry
    torch.cuda.synchronize()
    forward_error = repr(exc)[:4000]
    report["forward_traceback"] = traceback.format_exc()[-4000:]
    capture = dict((key, value) for key, value in capture.items() if key == "depth")
    with torch.no_grad():
        camera_feature = model.extract_camera_features(
            batch["img"],
            batch["points"],
            batch["camera2ego"],
            batch["lidar2ego"],
            batch["lidar2camera"],
            batch["lidar2image"],
            batch["camera_intrinsics"],
            batch["img_aug_matrix"],
            batch["lidar_aug_matrix"],
            metas,
        )
        lidar_feature = model.extract_lidar_features(batch["points"])
        fused = model.fuser([camera_feature, lidar_feature])
        decoded = model.decoder["backbone"](fused)
        decoded = model.decoder["neck"](decoded)
        pred = model.heads["object"](decoded, metas)
    torch.cuda.synchronize()
    trunk_only = True
    result = pred
elapsed = time.time() - started

for hook in hooks:
    hook.remove()

report["forward"] = {
    "full_forward_ok": full_forward_ok,
    "trunk_only_fallback": trunk_only,
    "error": forward_error,
    "seconds": round(elapsed, 2),
    "peak_gpu_mem_gb": round(torch.cuda.max_memory_allocated() / float(2 ** 30), 2),
}

# -------------------------------------------------------------- result/output
outputs = {}
try:
    if isinstance(result, (list, tuple)) and result and isinstance(result[0], dict):
        for index, entry in enumerate(result):
            item = {}
            if "boxes_3d" in entry:
                boxes = entry["boxes_3d"]
                tensor = getattr(boxes, "tensor", None)
                item["boxes_3d_count"] = int(tensor.shape[0]) if tensor is not None else None
                item["boxes_3d_stats"] = smoke_model.tensor_stats(tensor)
                item["boxes_3d_first5"] = (
                    [[round(float(v), 3) for v in row] for row in tensor[:5].tolist()]
                    if tensor is not None
                    else None
                )
            if "scores_3d" in entry:
                item["scores_3d_stats"] = smoke_model.tensor_stats(entry["scores_3d"])
                item["scores_3d_top5"] = [
                    round(float(v), 4) for v in entry["scores_3d"][:5].tolist()
                ]
            if "labels_3d" in entry:
                item["labels_3d_top5"] = [
                    int(v) for v in entry["labels_3d"][:5].tolist()
                ]
            outputs["sample_%d" % index] = item
    else:
        outputs["type"] = type(result).__name__
        outputs["length"] = len(result) if hasattr(result, "__len__") else None
except Exception as exc:  # noqa: BLE001
    outputs["decode_error"] = repr(exc)[:2000]
report["outputs"] = outputs

# ------------------------------------------------------------- depth channel
# Static evidence: the source lines that build the depth tensor and define what
# dtransform accepts. `BaseDepthTransform.forward` (inherited) builds the depth
# map; `DepthLSSTransform.__init__` defines `dtransform`.
class_lines = []
try:
    class_source = inspect.getsource(type(vtransform))
    class_lines = [
        line.strip()
        for line in class_source.splitlines()
        if "Conv2d(1, 8, 1)" in line or "self.dtransform = " in line or "get_cam_feats" in line
    ]
except Exception:  # noqa: BLE001
    pass
forward_lines = []
for cls in type(vtransform).__mro__:
    try:
        forward_lines.extend(inspect.getsource(cls.forward).splitlines())
    except Exception:  # noqa: BLE001
        continue
forward_lines = [
    line.strip()
    for line in forward_lines
    if "torch.zeros(batch_size, 6, 1" in line or "d = d.view(" in line or "dtransform(d)" in line
]

accepted_channels = int(dtransform[0].in_channels)
depth_stats = capture.get("depth") or {}
depth_channels = None
depth_spatial = None
if depth_stats.get("shape"):
    depth_channels = int(depth_stats["shape"][1])
    depth_spatial = list(depth_stats["shape"][-2:])
image_h, image_w = vtransform.image_size
# Strides of nn.Conv2d(8, 32, 5, stride=4, padding=2) then (32, 64, 5, stride=2, padding=2):
# the dtransform output must land exactly on the FPN feature size so that
# `torch.cat([d, x], dim=1)` in get_cam_feats works.
expected_h = conv_out(conv_out(image_h, 5, 4, 2), 5, 2, 2)
expected_w = conv_out(conv_out(image_w, 5, 4, 2), 5, 2, 2)
feature_size = list(vtransform.feature_size)
report["depth_check"] = {
    "dtransform_first_layer": str(dtransform[0]),
    "dtransform_accepted_channels": accepted_channels,
    "depth_tensor_shape": depth_stats.get("shape"),
    "depth_tensor_channels": depth_channels,
    "depth_tensor_nonzero_fraction": depth_stats.get("nonzero_fraction"),
    "depth_tensor_finite": smoke_model.finite_ok(depth_stats),
    "depth_tensor_min": depth_stats.get("min"),
    "depth_tensor_max": depth_stats.get("max"),
    "image_size": [image_h, image_w],
    "depth_spatial_size": depth_spatial,
    "depth_spatial_matches_image_size": depth_spatial == [image_h, image_w],
    "dtransform_output_size_computed": [expected_h, expected_w],
    "feature_size": feature_size,
    "dtransform_output_matches_feature_size": [expected_h, expected_w] == feature_size,
    "channels_match": depth_channels == accepted_channels,
    "V_D": int(vtransform.D),
    "V_C": int(vtransform.out_channels),
    "class_source_lines": class_lines,
    "forward_source_lines": forward_lines,
    "pass": bool(
        depth_channels is not None
        and depth_channels == accepted_channels
        and depth_spatial == [image_h, image_w]
        and [expected_h, expected_w] == feature_size
        and smoke_model.finite_ok(depth_stats)
    ),
}

# ------------------------------------------------------------------- infinite
stage_checks = {}
for name in ("depth", "camera_bev", "fused_bev", "decoder_neck"):
    stage_checks[name] = smoke_model.finite_ok(capture.get(name))
head_checks = dict(
    (name, smoke_model.finite_ok(stats))
    for name, stats in capture.items()
    if name.startswith("head")
)
output_checks = {}
for name, stats in (
    ("boxes_3d", outputs.get("sample_0", {}).get("boxes_3d_stats")),
    ("scores_3d", outputs.get("sample_0", {}).get("scores_3d_stats")),
):
    output_checks[name] = smoke_model.finite_ok(stats)
report["finite_check"] = {
    "stages": stage_checks,
    "head": head_checks,
    "outputs": output_checks,
    "captured": capture,
    "pass": bool(
        all(stage_checks.values())
        and all(head_checks.values())
        and all(output_checks.values())
    ),
}

# ------------------------------------------------------------------- verdict
report["pass"] = {
    "forward_no_shape_error": bool(full_forward_ok),
    "outputs_finite": report["finite_check"]["pass"],
    "depth_channels_match": report["depth_check"]["pass"],
    "checkpoint_ok": bool(ckpt_report["ok"]),
}
report["all_pass"] = all(report["pass"].values())

smoke_model.dump(os.path.join(art, "forward.json"), report)

print("batch shapes: " + json.dumps(batch_description["shapes"]))
print("forward: full=%s trunk_only=%s seconds=%.2f peak_mem=%.2fGB"
      % (full_forward_ok, trunk_only, report["forward"]["seconds"], report["forward"]["peak_gpu_mem_gb"]))
print("forward error: %s" % forward_error)
print("depth tensor: shape=%s channels=%s accepted_by_dtransform=%s"
      % (depth_stats.get("shape"), depth_channels, accepted_channels))
print("depth nonzero fraction: %s" % depth_stats.get("nonzero_fraction"))
print("depth spatial matches image size: %s (%s vs %s)"
      % (depth_spatial == [image_h, image_w], depth_spatial, [image_h, image_w]))
print("dtransform output size computed: %s feature_size=%s match=%s"
      % ([expected_h, expected_w], feature_size, [expected_h, expected_w] == feature_size))
print("depth channel pass: %s" % report["depth_check"]["pass"])
print("class source lines: %s" % json.dumps(class_lines))
print("forward source lines: %s" % json.dumps(forward_lines))
print("stage finite: " + json.dumps(stage_checks))
print("head finite: " + json.dumps(head_checks))
print("outputs: " + json.dumps(output_checks))
print("decoded boxes: %s" % json.dumps(outputs.get("sample_0", {}).get("boxes_3d_count")))
print("verdict: " + json.dumps(report["pass"]))
print("FORWARD_OK" if report["all_pass"] else "FORWARD_FAILED")
''',
}

for _name, _source in MODULES.items():
    _target = os.path.join(WORK if _name == "smoke_common.py" else SCRIPTS, _name)
    with open(_target, "w") as _handle:
        _handle.write(_source)
    # Fail here, loudly, instead of twenty minutes into a GPU build.
    compile(_source, _target, "exec")
    print("wrote %-24s %6d bytes" % (_name, len(_source)))

sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

print()
print("WORK    ", sc.WORK)
print("ROOT    ", sc.ROOT)
print("ENV     ", sc.ENV)
print("REPO    ", sc.REPO)
print("SCRIPTS ", sc.SCRIPTS)
print("ART     ", sc.ART)
print("LOGS    ", sc.LOGS)
print("pin     ", sc.PIN_SHORT)
print()
print(sc.capture("df -h /kaggle/working /root /tmp | sed 's/  */ /g'"))
print("dataset root:", sc.dataset_root())
print(sc.update(bootstrap={"work": sc.WORK, "root": sc.ROOT, "scripts": sc.SCRIPTS}))
print("BOOTSTRAP_OK")


In [ ]:
"""System probe: GPU, driver, CUDA toolkits, compilers, dataset mount.

Records what the build cell has to work with. No downloads, no installs.
"""

import glob
import json
import os
import re
import shutil
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

report = {"python": sys.version, "executable": sys.executable}
report["os_release"] = sc.capture("grep -E '^(NAME|VERSION)=' /etc/os-release")
report["kernel"] = sc.capture("uname -r")
report["cpu_count"] = os.cpu_count()
report["mem_total"] = sc.capture("free -g | sed -n 2p")
report["disk"] = sc.capture("df -h /kaggle/working /root | sed 's/  */ /g'")

report["nvidia_smi"] = sc.capture(
    "nvidia-smi --query-gpu=index,name,driver_version,memory.total,compute_cap "
    "--format=csv,noheader"
)

report["gcc_default"] = sc.capture("gcc --version 2>/dev/null | head -1")
report["gxx_default"] = sc.capture("g++ --version 2>/dev/null | head -1")
report["compilers"] = sc.capture("ls /usr/bin/gcc* /usr/bin/g++* 2>/dev/null | tr '\\n' ' '")
report["uv"] = shutil.which("uv") or sc.capture("command -v uv")
report["sys_python"] = sys.version.split()[0]

# --------------------------------------------------------------- CUDA toolkits
candidates = []
for line in sc.capture("ls -d /usr/local/cuda* 2>/dev/null").splitlines():
    line = line.strip()
    if line:
        candidates.append(line)
for extra in glob.glob("/usr/local/cuda*/bin/nvcc") + glob.glob("/usr/bin/nvcc") + glob.glob(
    "/usr/lib/cuda/bin/nvcc"
):
    candidates.append(os.path.dirname(os.path.dirname(extra)))
which_nvcc = shutil.which("nvcc")
if which_nvcc:
    candidates.append(os.path.dirname(os.path.dirname(which_nvcc)))

toolkits = []
for base in sorted(set(candidates)):
    nvcc = os.path.join(base, "bin", "nvcc")
    if not os.path.isfile(nvcc):
        continue
    raw = sc.capture('"%s" --version' % nvcc)
    match = re.search(r"release (\d+\.\d+)", raw)
    toolkits.append(
        {
            "path": base,
            "nvcc": nvcc,
            "release": match.group(1) if match else None,
            "raw": raw.splitlines()[-1] if raw else None,
            "cudart_header": os.path.isfile(os.path.join(base, "include", "cuda_runtime.h")),
        }
    )
report["cuda_toolkits_on_disk"] = toolkits
report["CUDA_HOME_env"] = os.environ.get("CUDA_HOME")
report["cuda_version_json"] = sc.capture("cat /usr/local/cuda*/version.json 2>/dev/null | head -20")
report["cuda_apt_sources"] = sc.capture(
    "grep -rh '^deb ' /etc/apt/sources.list /etc/apt/sources.list.d/ 2>/dev/null | grep -i cuda"
)
report["cuda_keyring"] = os.path.isfile("/usr/share/keyrings/cuda-archive-keyring.gpg")

# ------------------------------------------------------------------- dataset
root = sc.dataset_root()
report["dataset"] = {"root": root}
if root:
    report["dataset"]["version_dirs"] = sc.capture('ls "%s" | tr "\\n" " "' % root)
    infos = []
    for pattern in ("nuscenes_infos_train.pkl", "nuscenes_infos_val.pkl", "nuscenes_infos_test.pkl"):
        infos.append({"file": pattern, "exists": os.path.isfile(os.path.join(root, pattern))})
    report["dataset"]["infos"] = infos
    report["dataset"]["real_sample_possible"] = any(item["exists"] for item in infos)

print(json.dumps(report, indent=2, sort_keys=True))
sc.save_json("system.json", report)
sc.update(system=report, cuda_toolkits=toolkits)
print("PROBE_OK")


In [ ]:
"""Python 3.8 venv via `uv`.

Kaggle's system interpreter is 3.13 and the pinned wheels are cp38-only, so the
whole pinned stack lives in an isolated uv-managed interpreter. Idempotent: if
the venv already reports 3.8 this cell does nothing.
"""

import json
import os
import shutil
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

log = os.path.join(sc.LOGS, "venv.log")
report = {"env": sc.ENV, "python": sc.PY}

already, version = sc.venv_ok()
if already:
    report["skipped"] = "venv already at Python " + version
    print(report["skipped"])
else:
    uv = shutil.which("uv")
    if uv is None:
        rc, _ = sc.sh(
            '"%s" -m pip install -q uv' % sys.executable,
            log=log,
        )
        uv = shutil.which("uv") or "uv"
        print("uv:", uv)
    report["uv"] = sc.capture("%s --version" % uv)

    # 3.8.20 is what uv's python-build-standalone channel provides for cp38.
    rc, _ = sc.sh("%s python install 3.8.20" % uv, log=log)
    report["uv_python_install_rc"] = rc
    rc, _ = sc.sh('%s venv --python 3.8.20 "%s"' % (uv, sc.ENV), log=log)
    report["uv_venv_rc"] = rc
    ok, version = sc.venv_ok()
    if not ok:
        # Fallback: let uv resolve any 3.8 it has.
        rc, _ = sc.sh('%s venv --python 3.8 "%s"' % (uv, sc.ENV), log=log)
        report["uv_venv_38_any_rc"] = rc
        ok, version = sc.venv_ok()
    report["venv_ok"] = ok
    report["python"] = version

ok, version = sc.venv_ok()
report["final_python"] = version
report["final_ok"] = bool(ok)
print(json.dumps(report, indent=2, sort_keys=True))
sc.save_json("venv.json", report)
sc.update(venv=report)
assert ok, "could not create a Python 3.8 venv: see " + log
print("VENV_OK " + version)


In [ ]:
"""Install the pinned stack into the Python 3.8 venv.

torch 1.10.1+cu113 (PyTorch cu113 index), mmcv-full 1.4.0 (OpenMMLab cu113 /
torch1.10.0 wheel index), mmdet 2.20.0 (PyPI). Nothing is upgraded to fit a
newer CUDA: if these wheels cannot be installed the ticket stops here.

Idempotent: skips the install when the venv already reports the pinned versions.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

log = os.path.join(sc.LOGS, "pinned_wheels.log")
uv = "uv pip install --python " + sc.PY

probe = sc.capture(
    '"%s" -c "import torch, mmcv, mmdet; print(torch.__version__, mmcv.__version__, mmdet.__version__)"'
    % sc.PY
)
report = {"probe_before": probe}
already = probe.split() == ["1.10.1+cu113", "1.4.0", "2.20.0"]

commands = [
    uv + ' "setuptools==59.5.0" "wheel" "packaging==21.3" "ninja"',
    uv
    + " --index-url https://download.pytorch.org/whl/cu113"
    + " --extra-index-url https://pypi.org/simple"
    + ' "torch==1.10.1+cu113" "numpy==1.22.4"',
    uv
    + " -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html"
    + ' "mmcv-full==1.4.0"',
    uv
    + ' "mmdet==2.20.0" "matplotlib" "terminaltables" "pycocotools" "six" "scipy"'
    + ' "pyyaml" "numpy==1.22.4"',
]

report["skipped"] = already
if not already:
    report["steps"] = []
    for command in commands:
        rc, _ = sc.sh(command, log=log, timeout=3600)
        report["steps"].append({"command": command, "rc": rc})
        if rc != 0:
            print("STEP FAILED (rc=%d): %s" % (rc, command))
            print("see " + log)
            break

report["package_list"] = sc.capture('"%s" -m pip list 2>/dev/null || uv pip list --python "%s"' % (sc.PY, sc.PY))
rc, text = sc.run_script("step_env.py", log_name="step_env.log")
report["env_check_rc"] = rc
report["env_check_ok"] = "ENV_OK" in text
sc.save_json("pinned_wheels.json", report)
sc.update(pinned_wheels={"ok": report["env_check_ok"], "skipped": already})
print()
print("PACKAGE_LIST:")
print(report["package_list"])
print("PINNED_WHEELS_OK" if report["env_check_ok"] else "PINNED_WHEELS_FAILED (see %s)" % log)


In [ ]:
"""Clone MIT BEVFusion and check out the pinned commit `da93b27`.

Also records the static evidence for the pin: the release commit has neither
`add_depth_features` nor `height_expand`, its depth tensor is built as
`torch.zeros(batch_size, 6, 1, *image_size)` and `dtransform` starts with
`Conv2d(1, 8, 1)`.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

log = os.path.join(sc.LOGS, "clone.log")
report = {"repo": sc.REPO, "pin_short": sc.PIN_SHORT}

if not os.path.isdir(os.path.join(sc.REPO, ".git")):
    rc, _ = sc.sh('git clone --quiet "%s" "%s"' % (sc.REPO_URL, sc.REPO), log=log, timeout=1800)
    report["clone_rc"] = rc
else:
    report["clone_rc"] = "skipped (already cloned)"

rc, _ = sc.sh('git -C "%s" checkout --quiet %s' % (sc.REPO, sc.PIN_SHORT), log=log)
report["checkout_rc"] = rc
report["head"] = sc.capture('git -C "%s" rev-parse HEAD' % sc.REPO)
report["head_short"] = report["head"][:7]
report["log1"] = sc.capture('git -C "%s" log -1 --format="%H %ci %s"' % sc.REPO)
report["git_status"] = sc.capture('git -C "%s" status --porcelain' % sc.REPO)
report["branch"] = sc.capture('git -C "%s" rev-parse --abbrev-ref HEAD' % sc.REPO)

base_py = os.path.join(sc.REPO, "mmdet3d", "models", "vtransforms", "base.py")
depth_py = os.path.join(sc.REPO, "mmdet3d", "models", "vtransforms", "depth_lss.py")
report["depth_tensor_source"] = sc.capture('grep -n "torch.zeros(batch_size, 6, 1" "%s"' % base_py)
report["dtransform_source"] = sc.capture('grep -n "Conv2d(1, 8, 1)" "%s"' % depth_py)
report["add_depth_features_hits"] = int(
    sc.capture('grep -rn "add_depth_features" "%s" | wc -l' % sc.REPO) or 0
)
report["height_expand_hits"] = int(
    sc.capture('grep -rn "height_expand" "%s" | wc -l' % sc.REPO) or 0
)
report["config_exists"] = os.path.isfile(sc.CONFIG_PATH)

report["ok"] = bool(
    report["head"].startswith(sc.PIN_SHORT)
    and report["git_status"] == ""
    and report["add_depth_features_hits"] == 0
    and report["height_expand_hits"] == 0
    and report["config_exists"]
)
print(json.dumps(report, indent=2, sort_keys=True))
sc.save_json("pin.json", report)
sc.update(pin=report)
print("PIN_OK" if report["ok"] else "PIN_FAILED")


In [ ]:
"""Pick a CUDA toolkit that can compile extensions for torch 1.10.1+cu113.

torch 1.10.1 refuses to build a CUDA extension when the host `nvcc` major
version differs from its own (11.3): see
`torch/utils/cpp_extension.py::BuildExtension._check_cuda_version`.
Kaggle's image ships CUDA 12.x only, and NVIDIA's ubuntu2404 apt repo has no
11.x packages, so the preferred route is the **ubuntu2204** CUDA repo (same
apt key, 11.8 packages, runs fine on the noble userland).

Order of preference (recorded in the artifact, never silent):
  1. an existing CUDA 11.x toolkit on disk;
  2. `cuda-nvcc-11-8` + `cuda-cudart-dev-11-8` from NVIDIA's ubuntu2204 repo;
  3. the system CUDA 12.x `nvcc` with torch's major-version guard bypassed --
     explicit, logged deviation, and the build either works or is reported.
"""

import glob
import json
import os
import re
import shutil
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

log = os.path.join(sc.LOGS, "cuda_toolchain.log")
KEYRING = "/usr/share/keyrings/cuda-archive-keyring.gpg"
SOURCE_LIST = "/etc/apt/sources.list.d/cuda-ubuntu2204.list"
REPO_2204 = "https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/"


def nvcc_release(nvcc):
    raw = sc.capture('"%s" --version' % nvcc)
    match = re.search(r"release (\d+\.\d+)", raw)
    return (match.group(1) if match else None), raw.splitlines()[-1] if raw else ""


def discover():
    found = []
    bases = []
    for line in sc.capture("ls -d /usr/local/cuda* 2>/dev/null").splitlines():
        bases.append(line.strip())
    for nvcc in glob.glob("/usr/local/cuda*/bin/nvcc"):
        bases.append(os.path.dirname(os.path.dirname(nvcc)))
    which = shutil.which("nvcc")
    if which:
        bases.append(os.path.dirname(os.path.dirname(which)))
    for base in sorted(set(b for b in bases if b)):
        nvcc = os.path.join(base, "bin", "nvcc")
        if not os.path.isfile(nvcc):
            continue
        release, raw = nvcc_release(nvcc)
        found.append(
            {
                "path": base,
                "nvcc": nvcc,
                "release": release,
                "raw": raw,
                "cudart_header": os.path.isfile(os.path.join(base, "include", "cuda_runtime.h")),
            }
        )
    return found


report = {"attempts": [], "toolkits_initial": discover()}
chosen = None
for toolkit in report["toolkits_initial"]:
    if (toolkit["release"] or "").startswith("11."):
        chosen = toolkit
        chosen["strategy"] = "existing-11x"
        break

# ------------------------------------------------------- apt route: CUDA 11.8
if chosen is None:
    attempt = {"strategy": "apt-ubuntu2204-cuda-11-8"}
    if not os.path.isfile(KEYRING):
        sc.sh(
            "curl -fsSL %s | gpg --dearmor -o %s" % (REPO_2204 + "3bf863cc.pub", KEYRING),
            log=log,
        )
    with open(SOURCE_LIST, "w") as handle:
        handle.write("deb [signed-by=%s] %s /\n" % (KEYRING, REPO_2204))
    sc.sh("apt-get update -qq", log=log, timeout=1200)
    attempt["apt_policy"] = sc.capture("apt-cache policy cuda-nvcc-11-8 | head -6")
    rc, _ = sc.sh(
        "apt-get install -y -qq --no-install-recommends cuda-nvcc-11-8 cuda-cudart-dev-11-8",
        log=log,
        timeout=3600,
    )
    attempt["install_rc"] = rc
    candidates = []
    for base in glob.glob("/usr/local/cuda-11*"):
        nvcc = os.path.join(base, "bin", "nvcc")
        if os.path.isfile(nvcc):
            release, raw = nvcc_release(nvcc)
            candidates.append(
                {
                    "path": base,
                    "nvcc": nvcc,
                    "release": release,
                    "raw": raw,
                    "cudart_header": os.path.isfile(
                        os.path.join(base, "include", "cuda_runtime.h")
                    ),
                }
            )
    attempt["toolkits_found"] = candidates
    report["attempts"].append(attempt)
    for toolkit in candidates:
        if (toolkit["release"] or "").startswith("11."):
            toolkit["strategy"] = "apt-ubuntu2204-cuda-11-8"
            chosen = toolkit
            break

# ------------------------------------------------- system nvcc + guard bypass
bypass_guard = False
if chosen is None:
    toolkits = discover()
    report["attempts"].append({"strategy": "system-nvcc-with-guard-bypass", "toolkits": toolkits})
    if toolkits:
        # Highest available toolkit, with the documented guard bypass.
        chosen = sorted(toolkits, key=lambda item: item["release"] or "0")[-1]
        chosen["strategy"] = "system-nvcc-with-guard-bypass"
        bypass_guard = True

# ------------------------------------------------------------ host compiler
gcc = None
gxx = None
for candidate in ("gcc-11", "gcc-10", "gcc-9", "gcc-12", "gcc-13"):
    path = shutil.which(candidate)
    if path:
        gcc = path
        gxx = shutil.which(candidate.replace("gcc", "g++"))
        break
report["gcc_default"] = sc.capture("gcc --version 2>/dev/null | head -1")
report["gcc_chosen"] = gcc or "system default"

report["chosen"] = chosen
report["bypass_cuda_guard"] = bypass_guard
report["apt_source_list"] = sc.capture("cat %s 2>/dev/null" % SOURCE_LIST)
report["apt_keyring_present"] = os.path.isfile(KEYRING)
report["ok"] = chosen is not None

if chosen is not None:
    sc.update(
        build_toolchain={
            "cuda_home": chosen["path"],
            "cuda_release": chosen["release"],
            "strategy": chosen["strategy"],
            "bypass_cuda_guard": bypass_guard,
            "cc": gcc,
            "cxx": gxx,
            "cudart_header": chosen.get("cudart_header"),
        }
    )
sc.save_json("cuda_toolchain.json", report)
print(json.dumps(report, indent=2, sort_keys=True))
print("TOOLCHAIN_OK" if report["ok"] else "TOOLCHAIN_FAILED")


In [ ]:
"""Compile the mmdet3d CUDA ops from the pinned tree (`python setup.py develop`).

All eleven extensions are required: `import mmdet3d.ops` pulls every one of
them, and the model needs `bev_pool` (camera branch) and `voxel_layer` (LiDAR
branch). The build is done with the CUDA 11.x toolkit chosen by the previous
cell; compatibility flags are passed because the pinned torch predates both
newer libstdc++ headers and newer host compilers.

Idempotent: skipped when `import mmdet3d.ops` already works.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

BUILD_WRAPPER = '''"""Compile the pinned mmdet3d CUDA extensions (setup.py develop)."""

import os
import runpy
import sys

repo = os.environ["REPO"]

if os.environ.get("SMOKE_BYPASS_CUDA_GUARD") == "1":
    import torch.utils.cpp_extension as cpp

    def _bypassed(self):
        print(
            "[smoke] WARNING: torch CUDA major-version guard bypassed "
            "(nvcc major != torch.version.cuda major). Documented deviation."
        )

    cpp.BuildExtension._check_cuda_version = _bypassed

os.chdir(repo)
sys.argv = ["setup.py", "develop"]
runpy.run_path(os.path.join(repo, "setup.py"), run_name="__main__")
'''

build_log = os.path.join(sc.LOGS, "build_ops.log")
report = {}

toolchain = (sc.state().get("build_toolchain") or {})
if not toolchain.get("cuda_home"):
    raise RuntimeError("no CUDA toolchain recorded; run the cuda toolchain cell first")
report["toolchain"] = toolchain

already = sc.capture('"%s" -c "import mmdet3d.ops"' % sc.PY)
report["import_before"] = "already importable" if "Error" not in already and "Traceback" not in already else already[-2000:]
report["skipped"] = report["import_before"] == "already importable"

arch_probe = (
    '"' + sc.PY + '" -c "import torch; a = torch.cuda.get_device_capability(0);'
    ' print(str(a[0]) + chr(46) + str(a[1]))"'
)
capacity = sc.capture(arch_probe).strip()
if not capacity or "." not in capacity or capacity.startswith("ERROR"):
    capacity = "7.5"
arch = capacity
report["torch_cuda_arch_list"] = arch
report["gpu"] = sc.capture(
    "nvidia-smi --query-gpu=index,name,compute_cap --format=csv,noheader"
)

if not report["skipped"]:
    sc.write_script("build_ops.py", BUILD_WRAPPER)
    env = sc.py_env(
        CUDA_HOME=toolchain["cuda_home"],
        PATH=toolchain["cuda_home"] + "/bin:" + os.environ.get("PATH", ""),
        FORCE_CUDA="1",
        MAX_JOBS=str(max(1, min(4, os.cpu_count() or 2))),
        TORCH_CUDA_ARCH_LIST=arch,
        # torch 1.10 predates newer libstdc++/gcc: add the headers that newer
        # <limits>/<cstdint> users assume, and let nvcc accept a newer host gcc.
        CXXFLAGS="-O2 -include limits -include cstdint",
        NVCC_PREPEND_FLAGS="--allow-unsupported-compiler -include limits -include cstdint",
    )
    if toolchain.get("cc"):
        env["CC"] = toolchain["cc"]
    if toolchain.get("cxx"):
        env["CXX"] = toolchain["cxx"]
    if toolchain.get("bypass_cuda_guard"):
        env["SMOKE_BYPASS_CUDA_GUARD"] = "1"
    report["env"] = dict(
        (key, env[key])
        for key in (
            "CUDA_HOME",
            "FORCE_CUDA",
            "MAX_JOBS",
            "TORCH_CUDA_ARCH_LIST",
            "CXXFLAGS",
            "NVCC_PREPEND_FLAGS",
            "CC",
            "CXX",
            "SMOKE_BYPASS_CUDA_GUARD",
        )
        if key in env
    )
    report["nvcc"] = sc.capture('"%s" --version' % os.path.join(toolchain["cuda_home"], "bin", "nvcc"))
    rc, text = sc.sh(
        '"%s" "%s"' % (sc.PY, os.path.join(sc.SCRIPTS, "build_ops.py")),
        log=build_log,
        env=env,
        cwd=sc.REPO,
        timeout=7200,
    )
    report["build_rc"] = rc
    if rc != 0:
        tail = sc.capture('tail -120 "%s"' % build_log)
        print("BUILD LOG TAIL:")
        print(tail)
        report["build_log_tail"] = tail[-6000:]

rc, text = sc.run_script("step_ops_import.py", log_name="step_ops_import.log", timeout=1800)
report["ops_check_rc"] = rc
report["ops_ok"] = "OPS_OK" in text
sc.save_json("build_ops.json", report)
sc.update(build_ops={"ok": report["ops_ok"], "arch": arch, "skipped": report["skipped"]})
print("BUILD_OPS_OK" if report["ops_ok"] else "BUILD_OPS_FAILED (log: %s)" % build_log)


In [ ]:
"""Download the official fused detection checkpoint inside the kernel.

The notebook never downloads the checkpoint locally and never substitutes a
different model: it tries the URL named in the ticket first, records the exact
HTTP result, and only then falls back to the link the upstream BEVFusion README
lists today (same fused det checkpoint).

Idempotent: skipped when the file is already on disk with the recorded sha256.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

log = os.path.join(sc.LOGS, "checkpoint.log")
report = {"primary_url": sc.CKPT_URL_PRIMARY, "fallback_url": sc.CKPT_URL_FALLBACK}
previous = (sc.state().get("checkpoint") or {})

report["skipped"] = bool(
    os.path.isfile(sc.CKPT)
    and os.path.getsize(sc.CKPT) > 100 * 1024 * 1024
    and previous.get("sha256")
    and previous.get("size_bytes") == os.path.getsize(sc.CKPT)
)
if report["skipped"]:
    print("checkpoint already present: %s (%d bytes)" % (sc.CKPT, os.path.getsize(sc.CKPT)))
else:
    # Probe both URLs with a 1 KiB range request: fast, records the status code
    # without pulling 156 MB twice.
    probes = []
    for url in (sc.CKPT_URL_PRIMARY, sc.CKPT_URL_FALLBACK):
        out = sc.capture(
            'curl -sSL --max-time 120 -o /dev/null -r 0-1023 '
            '-w "http_code=%{http_code} bytes=%{size_download} effective=%{url_effective}" '
            + '"' + url + '"'
        )
        probes.append({"url": url, "probe": out})
    report["probes"] = probes
    print(json.dumps(probes, indent=2))

    served_by = None
    for entry in probes:
        if "http_code=200" in entry["probe"] or "http_code=206" in entry["probe"]:
            served_by = entry["url"]
            break
    report["served_by"] = served_by
    if served_by is None:
        raise RuntimeError("no checkpoint URL is reachable: " + json.dumps(probes))

    part = sc.CKPT + ".part"
    for attempt in (1, 2, 3):
        rc, _ = sc.sh(
            'curl -fSL --retry 3 --retry-delay 5 --max-time 3600 -o "%s" "%s"'
            % (part, served_by),
            log=log,
            timeout=3900,
        )
        report["download_attempts"] = attempt
        if rc == 0 and os.path.isfile(part) and os.path.getsize(part) > 100 * 1024 * 1024:
            break
    if not (os.path.isfile(part) and os.path.getsize(part) > 100 * 1024 * 1024):
        raise RuntimeError("checkpoint download failed from " + served_by + " (see " + log + ")")
    os.replace(part, sc.CKPT)

with open(sc.CKPT, "rb") as handle:
    magic = handle.read(4)
report["path"] = sc.CKPT
report["size_bytes"] = os.path.getsize(sc.CKPT)
report["sha256"] = sc.sha256(sc.CKPT)
report["magic_hex"] = magic.hex()
report["magic_is_torch_archive"] = magic[:2] == b"PK" or magic[:2] == b"\x80\x02"
report["ok"] = bool(
    report["size_bytes"] > 100 * 1024 * 1024 and report["magic_is_torch_archive"]
)
print(json.dumps(report, indent=2, sort_keys=True))
sc.save_json("checkpoint_file.json", report)
sc.update(checkpoint=report)
print("CHECKPOINT_OK" if report["ok"] else "CHECKPOINT_FAILED")


In [ ]:
"""Resolve the official config chain (no torchpack) and check its invariants.

Runs `smoke_cfg.py` in the pinned venv: merges `default.yaml` from every ancestor
directory of the leaf config (torchpack semantics), evaluates every `${...}`
expression, writes `artifacts/cfg_resolved.json` + `artifacts/config_snapshot.json`.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("smoke_cfg.py", log_name="smoke_cfg.log", timeout=600)
print()
print("rc = %d" % rc)
checks = sc.load_json("cfg_checks.json")
snapshot = sc.load_json("config_snapshot.json")
print("config chain:")
for entry in snapshot.get("chain", []):
    print("  %-90s %s" % (entry["path"], entry["sha256"][:16]))
print()
print("resolved values:")
for key in (
    "image_size",
    "point_cloud_range",
    "voxel_size",
    "vtransform_type",
    "vtransform_out_channels",
    "vtransform_dbound",
    "vtransform_feature_size",
    "fuser_in_channels",
    "lidar_sparse_shape",
    "head_test_grid_size",
):
    print("  %-28s %s" % (key, json.dumps(checks.get(key))))
print()
print("failed invariants: %s" % json.dumps(checks.get("failed")))
print("CONFIG_CELL_OK" if checks.get("ok") else "CONFIG_CELL_FAILED")


In [ ]:
"""Build the BEVFusion detection model from the resolved config.

Proves the config is self-consistent (camera 80ch + LiDAR 256ch -> ConvFuser 256,
decoder in 256, head in 512) and records parameter counts per submodule.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("step_model.py", log_name="step_model.log", timeout=1800)
report = sc.load_json("model.json")
print()
print("rc = %d" % rc)
print("params: %s" % json.dumps(report.get("model_params")))
print("init_cfg disabled (Swin pretrained download): %s" % report.get("init_cfg_disabled"))
print("init_cfg error: %s" % report.get("init_cfg_error"))
print("groups:")
for name in sorted(report.get("groups", {})):
    entry = report["groups"][name]
    print("  %-40s %-24s %12d" % (name, entry["type"], entry["params"]))
print("vtransform: %s" % json.dumps(report.get("vtransform"), sort_keys=True))
print("MODEL_CELL_OK" if "MODEL_OK" in text else "MODEL_CELL_FAILED (see logs/step_model.log)")


In [ ]:
"""Load the official fused checkpoint and list every key mismatch.

`step_checkpoint.py` builds the model, loads the checkpoint with
`load_state_dict(strict=False)` (a shape mismatch still raises, so it is caught
and reported), cross-checks with `mmcv.runner.load_checkpoint`, and verifies a
sample of weights really landed in the module.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("step_checkpoint.py", log_name="step_checkpoint.log", timeout=2400)
payload = sc.load_json("checkpoint.json")
ckpt = payload.get("checkpoint", {})
print()
print("rc = %d" % rc)
print("sha256        : %s" % ckpt.get("sha256"))
print("size_bytes    : %s" % ckpt.get("size_bytes"))
print("ckpt keys     : %s" % ckpt.get("ckpt_num_keys"))
print("model keys    : %s" % ckpt.get("model_num_keys"))
print("matched keys  : %s" % ckpt.get("matched_keys"))
print("shape mismatch: %s" % ckpt.get("shape_mismatch_count"))
print("missing keys  : %s (weight affecting: %s)"
      % (ckpt.get("missing_keys_total"), ckpt.get("missing_keys_weight_affecting_total")))
for key in ckpt.get("missing_keys", [])[:40]:
    print("   MISSING    " + key)
print("unexpected    : %s" % ckpt.get("unexpected_keys_total"))
for key in ckpt.get("unexpected_keys", [])[:40]:
    print("   UNEXPECTED " + key)
print("load_error    : %s" % ckpt.get("load_error"))
print("mmcv cross    : %s" % json.dumps(ckpt.get("cross_check_mmcv")))
print("weight sample : %s" % json.dumps(ckpt.get("weight_sample_check")))
print("CKPT_CELL_OK" if ckpt.get("ok") else "CKPT_CELL_FAILED (see logs/step_checkpoint.log)")


In [ ]:
"""Input batch: shapes, provenance, and why it is synthetic.

The Kaggle nuScenes mirror has no `nuscenes_infos_*.pkl` yet (open item of ticket
04), so no real sample can be pushed through the MIT pipeline. This cell records
that fact from the mounted dataset and prints the config-shaped synthetic batch
that the forward cell uses instead.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

system = sc.load_json("system.json")
dataset = system.get("dataset", {})
print("dataset root: %s" % dataset.get("root"))
print("infos: %s" % json.dumps(dataset.get("infos")))
print("real sample possible: %s" % dataset.get("real_sample_possible"))

rc, text = sc.run_script("smoke_batch.py", log_name="smoke_batch.log", timeout=900)
batch = sc.load_json("batch.json")
print()
print("rc = %d" % rc)
print("data_source: %s" % batch.get("data_source"))
print("why_synthetic: %s" % batch.get("why_synthetic"))
print("shapes:")
for key in sorted(batch.get("shapes", {})):
    print("  %-18s %s" % (key, json.dumps(batch["shapes"][key])))
sc.update(
    batch={
        "data_source": batch.get("data_source"),
        "shapes": batch.get("shapes"),
        "real_infos_available": bool(dataset.get("real_sample_possible")),
    }
)
print("BATCH_CELL_OK" if "BATCH_OK" in text else "BATCH_CELL_FAILED (see logs/smoke_batch.log)")


In [ ]:
"""Forward one sample: shapes, finiteness of every stage, depth-channel check.

Runs `step_forward.py` in the pinned venv. It hooks `dtransform` to capture the
exact depth tensor the transform receives, hooks the camera BEV, fused BEV,
decoder neck and head outputs, runs the model in eval mode under `no_grad`, and
records a pass/fail verdict per acceptance criterion.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("step_forward.py", log_name="step_forward.log", timeout=3600)
report = sc.load_json("forward.json")
print()
print("rc = %d" % rc)
print("batch shapes : %s" % json.dumps(report.get("batch", {}).get("shapes")))
print("forward      : %s" % json.dumps(report.get("forward")))
depth = report.get("depth_check", {})
print()
print("DEPTH CHECK")
for key in (
    "dtransform_first_layer",
    "dtransform_accepted_channels",
    "depth_tensor_shape",
    "depth_tensor_channels",
    "depth_tensor_nonzero_fraction",
    "depth_tensor_finite",
    "depth_tensor_min",
    "depth_tensor_max",
    "image_size",
    "depth_spatial_size",
    "depth_spatial_matches_image_size",
    "dtransform_output_size_computed",
    "feature_size",
    "dtransform_output_matches_feature_size",
    "channels_match",
    "V_D",
    "V_C",
    "class_source_lines",
    "forward_source_lines",
    "pass",
):
    print("  %-40s %s" % (key, json.dumps(depth.get(key))))
finite = report.get("finite_check", {})
print()
print("FINITE CHECK")
print("  stages : %s" % json.dumps(finite.get("stages")))
print("  head   : %s" % json.dumps(finite.get("head")))
print("  outputs: %s" % json.dumps(finite.get("outputs")))
print("  pass   : %s" % json.dumps(finite.get("pass")))
print()
print("OUTPUTS: %s" % json.dumps(report.get("outputs"), sort_keys=True))
print("VERDICT: %s" % json.dumps(report.get("pass"), sort_keys=True))
print("FORWARD_CELL_OK" if report.get("all_pass") else "FORWARD_CELL_FAILED (see logs/step_forward.log)")


In [ ]:
"""Aggregate every artifact into one pass/fail report.

Writes `artifacts/report.json` (machine-readable) and `smoke_test/REPORT.md`
(human-readable) into the notebook output, so the ticket can quote both.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

state = sc.state()
env = sc.load_json("env.json")
ops = sc.load_json("ops.json")
pin = sc.load_json("pin.json")
toolchain = sc.load_json("cuda_toolchain.json")
ckpt_file = sc.load_json("checkpoint_file.json")
cfg_checks = sc.load_json("cfg_checks.json")
model = sc.load_json("model.json")
ckpt = sc.load_json("checkpoint.json").get("checkpoint", {})
forward = sc.load_json("forward.json")

criteria = {
    "1_checkpoint_loads": bool(ckpt.get("ok")),
    "2_missing_unexpected_keys_listed": bool("missing_keys_total" in ckpt),
    "3_forward_no_shape_error": bool(forward.get("forward", {}).get("full_forward_ok")),
    "4_outputs_finite": bool(forward.get("finite_check", {}).get("pass")),
    "5_depth_channels_match": bool(forward.get("depth_check", {}).get("pass")),
    "6_report_saved": True,
}
supporting = {
    "env_pinned": bool(env.get("pinned_ok")),
    "ops_built_and_gpu_smoke": bool(ops.get("ok")),
    "pin_da93b27": bool(pin.get("ok")),
    "cuda_11x_toolkit": (toolchain.get("chosen") or {}).get("release"),
    "cuda_guard_bypassed": bool(toolchain.get("bypass_cuda_guard")),
    "checkpoint_obtained": bool(ckpt_file.get("ok")),
    "checkpoint_served_by": ckpt_file.get("served_by"),
    "config_invariants": bool(cfg_checks.get("ok")),
    "model_built": bool(model.get("model_params")),
}
report = {
    "ticket": "02-smoke-test-checkpoint",
    "pin": {"commit": pin.get("head"), "short": pin.get("head_short")},
    "env": env,
    "cuda_toolchain": toolchain.get("chosen"),
    "ops": {
        "ok": ops.get("ok"),
        "gpu_smoke": ops.get("gpu_smoke"),
    },
    "checkpoint": {
        "path": ckpt.get("path"),
        "sha256": ckpt.get("sha256"),
        "size_bytes": ckpt.get("size_bytes"),
        "served_by": ckpt_file.get("served_by"),
        "primary_url_probe": (ckpt_file.get("probes") or [{}])[0],
        "top_level_keys": ckpt.get("top_level_keys"),
        "ckpt_num_keys": ckpt.get("ckpt_num_keys"),
        "model_num_keys": ckpt.get("model_num_keys"),
        "matched_keys": ckpt.get("matched_keys"),
        "shape_mismatch_count": ckpt.get("shape_mismatch_count"),
        "shape_mismatch": ckpt.get("shape_mismatch"),
        "missing_keys_total": ckpt.get("missing_keys_total"),
        "missing_keys_weight_affecting_total": ckpt.get("missing_keys_weight_affecting_total"),
        "missing_keys": ckpt.get("missing_keys"),
        "unexpected_keys_total": ckpt.get("unexpected_keys_total"),
        "unexpected_keys": ckpt.get("unexpected_keys"),
        "load_error": ckpt.get("load_error"),
        "cross_check_mmcv": ckpt.get("cross_check_mmcv"),
        "weight_sample_check": ckpt.get("weight_sample_check"),
    },
    "batch": forward.get("batch"),
    "forward": forward.get("forward"),
    "depth_check": forward.get("depth_check"),
    "finite_check": {
        "stages": forward.get("finite_check", {}).get("stages"),
        "head": forward.get("finite_check", {}).get("head"),
        "outputs": forward.get("finite_check", {}).get("outputs"),
        "pass": forward.get("finite_check", {}).get("pass"),
    },
    "outputs": forward.get("outputs"),
    "criteria": criteria,
    "supporting": supporting,
    "all_pass": all(criteria.values()) and all(
        value for key, value in supporting.items() if key in ("env_pinned", "ops_built_and_gpu_smoke", "pin_da93b27", "checkpoint_obtained", "config_invariants")
    ),
}
sc.save_json("report.json", report)

lines = []
lines.append("# Smoke test report — pinned MIT BEVFusion (%s)" % (pin.get("head_short") or "?"))
lines.append("")
lines.append("- env: python %s, torch %s (cuda %s), mmcv-full %s, mmdet %s, numpy %s"
             % (env.get("python_version"), env.get("torch"), env.get("torch_cuda_build"),
                env.get("mmcv"), env.get("mmdet"), env.get("numpy")))
lines.append("- gpu: %s" % json.dumps(env.get("device_0")))
lines.append("- cuda toolkit for the build: %s" % json.dumps(toolchain.get("chosen")))
lines.append("- checkpoint: %s (%s bytes) sha256 %s served by %s"
             % (ckpt.get("path"), ckpt.get("size_bytes"), ckpt.get("sha256"), ckpt_file.get("served_by")))
lines.append("- config chain: %s" % ", ".join(os.path.basename(item["path"]) for item in sc.load_json("config_snapshot.json").get("chain", [])))
lines.append("")
lines.append("## Acceptance criteria")
for key in sorted(criteria):
    lines.append("- [%s] %s" % ("x" if criteria[key] else " ", key))
lines.append("")
lines.append("## Checkpoint keys")
lines.append("- checkpoint keys: %s, model keys: %s, matched: %s"
             % (ckpt.get("ckpt_num_keys"), ckpt.get("model_num_keys"), ckpt.get("matched_keys")))
lines.append("- shape mismatches: %s" % ckpt.get("shape_mismatch_count"))
lines.append("- missing: %s (weight affecting %s)" % (ckpt.get("missing_keys_total"), ckpt.get("missing_keys_weight_affecting_total")))
lines.append("- unexpected: %s" % ckpt.get("unexpected_keys_total"))
lines.append("")
lines.append("## Forward")
lines.append("- data source: %s (%s)" % (forward.get("batch", {}).get("data_source"), forward.get("batch", {}).get("why_synthetic")))
lines.append("- full forward ok: %s" % forward.get("forward", {}).get("full_forward_ok"))
lines.append("- seconds: %s, peak GPU memory: %s GB" % (forward.get("forward", {}).get("seconds"), forward.get("forward", {}).get("peak_gpu_mem_gb")))
lines.append("- depth tensor: %s" % json.dumps(forward.get("depth_check", {}).get("depth_tensor_shape")))
lines.append("- dtransform accepts %s channels, depth provides %s"
             % (forward.get("depth_check", {}).get("dtransform_accepted_channels"),
                forward.get("depth_check", {}).get("depth_tensor_channels")))
lines.append("- finite: stages=%s outputs=%s"
             % (json.dumps(forward.get("finite_check", {}).get("stages")),
                json.dumps(forward.get("finite_check", {}).get("outputs"))))
lines.append("")
with open(os.path.join(WORK, "REPORT.md"), "w") as handle:
    handle.write("\n".join(lines))

print("\n".join(lines))
print()
print("SUMMARY support: %s" % json.dumps(supporting, sort_keys=True))
print("SUMMARY criteria: %s" % json.dumps(criteria, sort_keys=True))
print("ALL_PASS" if report["all_pass"] else "NOT_ALL_PASS")


## Reading the result

Artifacts land in `/kaggle/working/smoke_test/artifacts` (small JSON, safe to
download with `kaggle kernels output`) and logs in `.../logs`.

| artifact | what it proves |
|---|---|
| `system.json` | GPU, driver, which CUDA toolkits exist, dataset mount |
| `venv.json`, `pinned_wheels.json`, `env.json` | the pinned stack is really Python 3.8 + torch 1.10.1+cu113 + mmcv-full 1.4.0 + mmdet 2.20.0 |
| `pin.json` | clone is at `da93b27`, tree clean, no `add_depth_features` / `height_expand` |
| `cuda_toolchain.json` | which `nvcc` compiled the ops, and whether torch's major-version guard was bypassed |
| `build_ops.json`, `ops.json` | every extension imports **and** `bev_pool` + hard voxelization ran on the T4 |
| `checkpoint_file.json` | URL that actually served the checkpoint, size, sha256, torch-archive magic |
| `config_snapshot.json`, `cfg_resolved.json`, `cfg_checks.json` | exact config chain + sha256, resolved values, invariants |
| `model.json` | the model builds from that config; parameter counts per branch |
| `checkpoint.json` | missing / unexpected / shape-mismatched keys, mmcv cross-check |
| `batch.json` | input shapes (synthetic, config-shaped) and the reason |
| `forward.json` | full forward result, per-stage finiteness, depth tensor channels vs `dtransform` |
| `report.json` | all of the above plus pass/fail per acceptance criterion |

### If a cell fails

* **pinned wheels** — read `logs/pinned_wheels.log`; a missing cp38 wheel is a
  hard stop for the ticket (the pin cannot be built).
* **cuda toolchain** — `cuda_toolchain.json` records the three strategies tried.
  If it ended in `system-nvcc-with-guard-bypass`, the compile is still torch
  1.10.1+cu113, but nvcc is 12.x; the build log says whether the CUDA runtime ABI
  held.
* **build ops** — `logs/build_ops.log` is the full compiler output; the cell
  prints its tail. The failing extension name is in `build_ops.json`.
* **forward** — `logs/step_forward.log` plus `forward.json:forward_traceback`
  carry the exact stack; `forward.trunk_only_fallback` tells you whether only the
  head failed.

### What this ticket deliberately does not do

* No real nuScenes sample: infos do not exist yet (ticket 04 open item), so the
  forward uses a synthetic config-shaped batch and says so everywhere.
* No evaluation, no training, no spray transform — those are later tickets.
* No substitution of another BEVFusion implementation or a newer framework
  version: if the pinned stack cannot be built, the report says exactly where.
